> **Versión t70:** copia de `02_rollout_moondream2.ipynb` con figuras de 70 px. Usa `conjunto_t70.zip` y guarda en `salidas/02_rollout_t70/`; la versión de 35 px no cambia.

# 02 · Attention rollout en el encoder visual de moondream2

Calcula el *attention rollout* (Abnar y Zuidema, 2020) del encoder visual sobre:
- las **20 imágenes** de la muestra (`muestra_mapas.csv`: 7 aciertos, 7 errores cercanos, 6 errores lejanos), en su versión original `c0`;
- **2 controles**: una imagen completamente gris (el fondo, sin figuras) y una de ruido.

El encoder **no recibe la pregunta**, así que el mapa es el mismo para cualquier pregunta sobre una imagen. Por eso se calcula una vez por imagen.

Además, se guarda la **norma** de cada token en cada capa. Los *attention sinks* suelen ser tokens de norma muy alta (Darcet et al., 2024).

Detalle técnico: el encoder se reimplementa línea por línea (`vision_encoder` de `vision.py`, revisión 2025-06-21), con la atención calculada de forma explícita (softmax(QKᵀ/√d)) en vez del kernel fusionado. Una celda verifica que la salida es igual a la del modelo original.

**Cómo correrlo:** igual que el 01. T4 GPU → *Ejecutar todas* → si se reinicia, *Ejecutar todas* otra vez. Tarda ~5–10 min. Los resultados quedan en `memoria_explicabilidad/salidas/02_rollout_t70/`.

In [ ]:
import importlib.metadata, os
VERSION = "4.46.3"
if importlib.metadata.version("transformers") != VERSION:
    os.system(f"pip install -q transformers=={VERSION}")
    print("Instalado. La sesión se reinicia: vuelve a 'Ejecutar todas'.")
    os.kill(os.getpid(), 9)
print("transformers", VERSION, "OK")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import glob, zipfile
BASE = "/content/drive/MyDrive/memoria_explicabilidad"
SALIDA = f"{BASE}/salidas/02_rollout_t70"
os.makedirs(f"{SALIDA}/png", exist_ok=True)

encontrados = sorted(glob.glob(f"{BASE}/**/conjunto_t70*.zip", recursive=True))
assert encontrados, "No encontré conjunto_t70.zip dentro de memoria_explicabilidad/"
with zipfile.ZipFile(encontrados[0]) as z:
    z.extractall("/content")
CONJUNTO = "/content/conjunto_t70"
print("Conjunto:", encontrados[0])

In [3]:
import torch
from transformers import AutoModelForCausalLM

MODEL_ID = "vikhyatk/moondream2"
MODEL_REVISION = "2025-06-21"
assert torch.cuda.is_available(), "Sin GPU: Entorno de ejecución → Cambiar tipo de entorno → T4 GPU"
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, trust_remote_code=True,
    torch_dtype=torch.bfloat16, device_map={"": "cuda"})
md = model.model   # el modelo interno (MoondreamModel) con .vision y .config
print("GPU:", torch.cuda.get_device_name(0))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


GPU: Tesla T4


In [4]:
# Módulos de código del modelo, cargados por trust_remote_code
import sys
def buscar(sufijo, atributo):
    for nombre, mod in list(sys.modules.items()):
        if mod is not None and "moondream" in nombre.lower() and nombre.endswith(sufijo) and hasattr(mod, atributo):
            return mod
    raise RuntimeError(f"No encontré el módulo {sufijo} con {atributo}")
vision_mod = buscar(".vision", "vision_encoder")
layers_mod = buscar(".layers", "linear")
CFG = md.config.vision
print("Capas:", CFG.enc_n_layers, "| cabezas:", CFG.enc_n_heads, "| tokens por recorte:", (CFG.crop_size // CFG.enc_patch_size) ** 2)

Capas: 27 | cabezas: 16 | tokens por recorte: 729


In [5]:
import torch.nn.functional as F

def atencion_explicita(x, w, n_heads):
    """Igual que layers.attn, y además devuelve la matriz de atención."""
    bsz, q_len, d_model = x.shape
    hd = d_model // n_heads
    q, k, v = [t.view(bsz, q_len, n_heads, hd).transpose(1, 2)
               for t in layers_mod.linear(x, w.qkv).chunk(3, dim=-1)]
    # La salida usa el mismo F.scaled_dot_product_attention que el original: así el encoder es idéntico.
    out = F.scaled_dot_product_attention(q, k, v).transpose(1, 2).reshape(bsz, q_len, d_model)
    # Aparte, la matriz softmax(QK^T/sqrt(d)) en float32, solo para el rollout.
    a = ((q.float() @ k.float().transpose(-2, -1)) / hd ** 0.5).softmax(dim=-1)
    return layers_mod.linear(out, w.proj), a

def encoder_instrumentado(crops):
    """Reimplementa vision_encoder() y calcula el rollout del recorte 0 (la vista global de la imagen)."""
    w = md.vision
    x = vision_mod.create_patches(crops, CFG.enc_patch_size)
    x = w.patch_emb(x) + w.pos_emb
    n = x.shape[1]
    eye = torch.eye(n, device=x.device)
    rollout, normas = eye.clone(), []
    for block in w.blocks:
        salida, a = atencion_explicita(vision_mod.layer_norm(x, block.ln1), block.attn, CFG.enc_n_heads)
        x = x + salida
        x = x + vision_mod.mlp(vision_mod.layer_norm(x, block.ln2), block.mlp)
        capa = a[0].mean(dim=0) + eye                       # promedio de cabezas + identidad (conexión residual)
        rollout = (capa / capa.sum(dim=-1, keepdim=True)) @ rollout
        normas.append(x[0].float().norm(dim=-1))            # norma de cada token después de la capa
        ultima = a[0].mean(dim=0)
    x = vision_mod.layer_norm(x, w.post_ln)
    return x, dict(rollout=rollout.cpu(), normas=torch.stack(normas).cpu(), ultima=ultima.cpu())

In [6]:
# Verificación: la reimplementación da la misma salida que el encoder original
from PIL import Image
img = Image.open(f"{CONJUNTO}/imagenes/q002_c0.png").convert("RGB")
crops, _ = vision_mod.prepare_crops(img, CFG, device=md.device)
with torch.inference_mode():
    original = vision_mod.vision_encoder(crops, md.vision, CFG).float()
    propia, _ = encoder_instrumentado(crops)
dif = (original - propia.float()).abs().max().item()
cos = F.cosine_similarity(original.flatten(0, 1), propia.float().flatten(0, 1), dim=-1).min().item()
print(f"Diferencia máxima: {dif:.4f} | similitud coseno mínima entre tokens: {cos:.5f} (1 = idénticos)")
assert cos > 0.99, "La reimplementación no coincide con el encoder original: no sigas, avísame."

Diferencia máxima: 0.0000 | similitud coseno mínima entre tokens: 1.00000 (1 = idénticos)


In [8]:
import numpy as np, csv, json
import matplotlib.pyplot as plt

MUESTRA = {'q040': 'acierto', 'q022': 'acierto', 'q018': 'acierto', 'q007': 'acierto', 'q108': 'acierto', 'q016': 'acierto', 'q074': 'acierto', 'q054': 'cercano', 'q010': 'cercano', 'q085': 'cercano', 'q017': 'cercano', 'q011': 'cercano', 'q093': 'cercano', 'q089': 'cercano', 'q113': 'lejano', 'q078': 'lejano', 'q056': 'lejano', 'q029': 'lejano', 'q028': 'lejano', 'q079': 'lejano'}   # t70: PEGAR aquí el diccionario de 20 preguntas que imprime el paso B6
assert MUESTRA, "Falta pegar MUESTRA: ver el paso B6 (muestra_mapas.csv de 01_conducta_t70)"
preguntas = {p["qid"]: p for p in csv.DictReader(open(f"{CONJUNTO}/preguntas.csv"))}

imagenes = {q: Image.open(f"{CONJUNTO}/imagenes/{q}_c0.png").convert("RGB") for q in MUESTRA}
imagenes["gris"] = Image.new("RGB", (1024, 1024), (200, 200, 200))
imagenes["ruido"] = Image.fromarray(np.random.default_rng(0).integers(0, 256, (1024, 1024, 3), dtype=np.uint8))

LADO = CFG.crop_size // CFG.enc_patch_size   # 27
mapas, filas = {}, []
for ident, img in imagenes.items():
    crops, _ = vision_mod.prepare_crops(img, CFG, device=md.device)
    with torch.inference_mode():
        _, r = encoder_instrumentado(crops)
    recibido = r["rollout"].mean(dim=0).numpy()       # cuánto aporta cada parche a todos los demás (igual que el piloto v3)
    norma_final = r["normas"][-1].numpy()
    mapas[f"{ident}_rollout"] = r["rollout"].numpy().astype(np.float16)
    mapas[f"{ident}_recibido"] = recibido
    mapas[f"{ident}_ultima_capa"] = r["ultima"].mean(dim=0).numpy()
    mapas[f"{ident}_normas"] = r["normas"].numpy().astype(np.float16)

    top_r = np.argsort(recibido)[::-1][:5]
    top_n = np.argsort(norma_final)[::-1][:5]
    filas.append(dict(id=ident, grupo=MUESTRA.get(ident, "control"),
                      pregunta=preguntas[ident]["pregunta"] if ident in preguntas else "",
                      top5_rollout=" ".join(map(str, top_r)), top5_norma=" ".join(map(str, top_n)),
                      norma_max_sobre_mediana=round(float(norma_final.max() / np.median(norma_final)), 2)))

    fig, ax = plt.subplots(1, 3, figsize=(12, 4.2))
    ax[0].imshow(img); ax[0].set_title(f"{ident} ({MUESTRA.get(ident, 'control')})")
    for eje, m, t in [(ax[1], recibido, "rollout"), (ax[2], norma_final, "norma del token (última capa)")]:
        eje.imshow(img); eje.imshow(m.reshape(LADO, LADO), cmap="jet", alpha=0.55,
                                     extent=(0, img.width, img.height, 0), interpolation="nearest")
        eje.set_title(t)
    for eje in ax: eje.axis("off")
    if ident in preguntas: fig.suptitle(preguntas[ident]["pregunta"], fontsize=9)
    plt.tight_layout(); plt.savefig(f"{SALIDA}/png/{ident}.png", dpi=110); plt.close(fig)
    print("OK", ident, "| top-5 rollout:", top_r, "| top-5 norma:", top_n)

np.savez_compressed(f"{SALIDA}/mapas_rollout.npz", **mapas)
with open(f"{SALIDA}/resumen.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(filas[0])); w.writeheader(); w.writerows(filas)

OK q040 | top-5 rollout: [161 135 188 721  23] | top-5 norma: [513 540  23  22 303]
OK q022 | top-5 rollout: [161 135 188 721  23] | top-5 norma: [513 540  23  22 623]
OK q018 | top-5 rollout: [161 135 188 721  23] | top-5 norma: [513 540  23  22 411]
OK q007 | top-5 rollout: [161 135 188 721  23] | top-5 norma: [513 540  23  22 687]
OK q108 | top-5 rollout: [161 135 188 721  23] | top-5 norma: [513 540  23  22 708]
OK q016 | top-5 rollout: [161 135 188 721  23] | top-5 norma: [513 540  23  22 651]
OK q074 | top-5 rollout: [161 135 188 721  23] | top-5 norma: [513 540  23  22 723]
OK q054 | top-5 rollout: [161 135 188 721 705] | top-5 norma: [513 540  23  22 411]
OK q010 | top-5 rollout: [161 135 188 721  23] | top-5 norma: [513 540  23  22 722]
OK q085 | top-5 rollout: [161 135 188 721  23] | top-5 norma: [513 540  23  22 668]
OK q017 | top-5 rollout: [161 135 188 721  23] | top-5 norma: [513 540  23  22 722]
OK q011 | top-5 rollout: [161 135 188 721  23] | top-5 norma: [513 540  23  

In [9]:
import transformers, platform
info = dict(modelo=MODEL_ID, revision=MODEL_REVISION, transformers=transformers.__version__, torch=torch.__version__,
            gpu=torch.cuda.get_device_name(0), python=platform.python_version(),
            verificacion_dif_max=dif, verificacion_coseno_min=cos, n_imagenes=len(imagenes),
            vram_pico_mb=round(torch.cuda.max_memory_allocated() / 1024**2))
json.dump(info, open(f"{SALIDA}/info_corrida.json", "w"), indent=2)

g = set(filas[-2]["top5_rollout"].split())   # control gris
coincide = [len(g & set(f["top5_rollout"].split())) for f in filas[:-2]]
print("Puntos calientes del rollout compartidos con la imagen gris (de 5), por imagen:", coincide)
print("\nLISTO. Archivos en", SALIDA, ":", sorted(os.listdir(SALIDA)))

Puntos calientes del rollout compartidos con la imagen gris (de 5), por imagen: [5, 5, 5, 5, 5, 5, 5, 4, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5]

LISTO. Archivos en /content/drive/MyDrive/memoria_explicabilidad/salidas/02_rollout_t70 : ['info_corrida.json', 'mapas_rollout.npz', 'png', 'resumen.csv']
